# Récupérer des données (Partie 2) - Bases 3 - 09/03/2026

Dans ce notebook, nous verrons comment récupérer des informations depuis l'API d'un musée (Victoria and Albert Museum) et l'accès à leur serveur IIIF.

Comme toujours, toutes les cellules peuvent être executées dans l'ordre, mais essayez de prédire la sortie de chaque cellule avant de l'exécuter.
N'hésitez pas à jouer avec le code, le bidouiller pour tester des alternatives — c'est l'un des grands avantages d'utiliser un notebook.

In [ ]:
import requests
import pandas as pd

from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

### 1.1. Quelques requêtes de base

Pour choisir un objet manuellement, et extraire ses métadonnées, nous pouvons exécuter la requête suivante :

In [ ]:

req = requests.get('https://api.vam.ac.uk/v2/museumobject/O828146')
object_data = req.json()
print("The object you requested has the title '%s'" % object_data["record"]["titles"][0]["title"])

The object you requested has the title 'Picturesque Holland'


❓ En explorant le dictionnaire JSON `object_data`, affichez: 
- son année de production
- son numéro d'archive ('Accession Number')
- sa technique principale
- sa position dans le musée

In [3]:
# Votre réponse ici

❓ En guide de 'méta-résultat', affichez aussi le nombre de résultats obtenus pour cette requête :

In [49]:
# Votre réponse ici

Essayons maintenant de faire une recherche par nom.

Le principe d'une API est de pouvoir contenir dans une seule requête, en modifiant l'URL de requête. 

Par exemple, pour récupérer un CSV des résultats de "Napoléon", il suffit de lancer : 

`https://api.vam.ac.uk/v2/objects/search?q=Napoleon&response_format=csv`


❗ Remarquez :
- que l'on spécifie dans l'URL le format CSV
- que Pandas est capable de lire directement le résultat d'une requête web

In [5]:
object_df = pd.read_csv("https://api.vam.ac.uk/v2/objects/search?q=Napoleon&response_format=csv")
object_df.head()

,accessionNumber,accessionYear,systemNumber,objectType,_primaryTitle,_primaryPlace,_primaryMaker__name,_primaryMaker__association,_primaryDate,_primaryImageId,_sampleMaterial,_sampleTechnique,_sampleStyle,_currentLocation__displayName,_objectContentWarning,_imageContentWarning
0,A.17-1948,1948,O123417,Bust,Napoleon I,Carrara,"Chaudet, Antoine-Denis",after,1807-1809,2013GV6443,carrara marble,carved,NaN,"Europe 1600-1815, Room 1",False,False
1,A.43-1983,1983,O34920,Bust,Napoleon III,Paris,"Carpeaux, Jean-Baptiste",sculptor,1874,2017JY6250,marble,sculpted,NaN,in store,False,False
2,86-1874,1874,O148954,Figure,Napoleon as Emperor,Staffordshire,Unknown,NaN,ca. 1810-1820,2007BP1353,earthenware,NaN,NaN,"Ceramics, Room 138, The Harry and Carol Djanog...",False,False
3,A.16-1984,1984,O313248,Bust,Napoleon,France,"Houdon, Jean-Antoine",after,ca. 1806,2014GW6694,bronze,cast,NaN,In Store,False,False
4,P.8-1926,1926,O1225585,Drawing,Napoleon after death,Saint Helena,Marryat,artist,ca. 1821,2017JV6803,Ink,wash,NaN,"Prints & Drawings Study Room, level H",False,False


❓ Modifiez la cellule précédente pour obtenir les résultats de 5 à 10 quand on cherche "Genève"

In [102]:
# Votre réponse ici

### 1.2. Filtrer par valeur de champ

Plutôt qu'une recherche par mot-clé, il est parfois plus utile de chercher par champ. Par exemple, le matériau, la date de création, le lieu, etc. 

Là encore, tout passe par l'URL (cf. cours).

Les champs existants incluent, entre autres : 
- id_material (eg. AAT10797 pour le verre)
- q_object_title
- q_place_name 
- id_place (eg. x28980 pour Londres)
- q_material_technique
- q_actor
- kw_object_type
- kw_accession_number
- kw_accession_year
- year_made_from
- year_made_to
- on_display (eg. south_kensington)
- images_exist


❓ Quelle est la différence, par exemple entre `q_place_name` et `id_place` ?

❓ Quel est le type de données, par exemple, de l'année *au sein de la requête* ? Et de *images_exist*?


*Votre réponse ici*

❓ Écrivez une requête qui vous renvoie les objects venant de Rome et en papier. 

Aidez-vous pour cela d'une courte [liste des identifiants les plus connus](https://developers.vam.ac.uk/guide/v2/common-identifiers.html)

In [31]:
# Votre réponse ici

Courte digression, mais qui vous fera gagner en lisibilité : 

❓ Écrivez une fonction qui prend en entrée une chaîne de caractères contenant votre requête -- sans les parties redondantes -- et qui renvoie une `DataFrame`.

In [28]:
# Votre réponse ici

Il est possible de refuser certaines valeurs (filtres négatifs), en ajoutant un "-" devant la valeur du champ.

❓ Écrivez la requête qui récupère les objets qui contiennent du verre (AAT10797) mais **pas** de plastique (AAT14570).

In [32]:
# Votre réponse ici

### 1.3. Une difficulté fréquente : la taille de la page

Il est rarissime (et c'est tant mieux) qu'une requête API vous renvoie *tous* les résultats disponibles. Vous recontrerez plus souvent une logique de "page Google", avec un certain nombre de pages et un certain nombre de résultats par page.

Si vous voulez tout récolter, à vous de prendre ceci en compte en jouant avec les paramètres, et les requêtes.

- page_size (jusqu`à 100)
- page

❓ Comparez par exemple les résultats aux pages 1 et 10 de la requête suivante : "https://api.vam.ac.uk/v2/objects/search?q=switzerland&response_format=csv&page_size=100"


In [101]:
# Votre réponse ici

❓ Écrivez une fonction qui itère sur les pages de résultats pour la simple requête "geneva", et vous renvoie une `DataFrame` complète.

In [42]:
# Votre réponse ici

### Exercice 1

À partir de l'API du V&A, récupérez tous les objets de leur collection qui proviennent de Paris, entre 1750 et 1760, et dont les objets sont affichés au musée de South Kensigton.

❓ Combien d'objets cela représente-t'il ?
❓ En quelle année le musée a-t'il acquis le plus de ces objets ?

Produisez les histogrammes suivants:
- Un histogramme des types d'objets
- Un histogramme des matériaux
- Un histogramme des lieux de provenances

In [76]:
# Votre réponse ici

# 2. IIIF

Dans la majorité des cas récents, les musées stockent leurs images sur des serveurs IIIF. C'est le cas du V&A.

Reprenons donc notre exemple, et essayons de récupérer les images.

❓ Pour cela, modifiez la requête pour n'inclure que les objets ayant une image.


In [83]:
# Votre réponse ici

❓ Quelle colonne vous indique où trouver l'image ?

❓ Modifiez la dataframe pour ajouter deux colonne de liens vers le serveur IIIF:
- Une pour le lien vers le manifeste (eg. 'https://framemark.vam.ac.uk/collections/2006AN7529/info.json')
- Une pour le lien vers l'image (à vous de trouver ;))

❗ Attention, il existe deux manifestes : un pour l'objet et un pour l'image. À vous de choisir.

In [84]:
# Votre réponse ici

### Exercice 2

Prenez une image (par exemple la première) et affichez là via Python en trois versions (si possible côte à côte):
- La version en pleine résolution
- Une version en résolution 100,100
- Une version pivotée de 90 dégrés


Rappel, pour ouvrir une image via une URL: 

````
response = requests.get(URL)
img = Image.open(BytesIO(response.content))
plt.imshow(img)


In [98]:
# Votre réponse ici

### Exercice 3

Cas typique de ce qui pourrait être le début de votre projet de recherche, constitutez votre propre corpus. Continuons de prendre ici l'exemple du V&A, avec les objets parisiens entre 1750 et 1760.

Pour cela :
- Triez votre `DataFrame` par ordre alphabétique d'artiste
- Ajoutez dans votre `DataFrame` la résolution de chaque photographie de l'objet
- Pour les 10 premières images, et téléchargez sur votre machine (dans un dossier spécifique) :
  - Le manifeste JSON de l'objet
  - Une version en haute-résolution couleur de l'objet en PNG
  - Une version grise en basse résolution en JPG
  - ❗ Faites bien attention à garder trace de quel objet il s'agit dans le nom du fichier
- Ajoutez le chemin (path) vers l'image numérique haute-résolution et couleur dans la `DataFrame`
- Exportez ceci en CSV.

In [100]:
# Votre réponse ici